In [103]:
import numpy as np
f_max = 20000

def furier_projection(t_chop, data, n):
    t_points = len(data)
    dt = t_chop/t_points
    calc = 0.0
    omega = 2*np.pi/t_chop
    t_global = 0.0
    t = np.arange(t_points) * dt
    
    # for d in data:
    #     calc += d*dt*np.exp(-1j*omega*n*t_global)
    #     t_global += dt
    
    # return calc/t_chop
    return np.sum(data * np.exp(-1j * omega * n * t)) * dt / t_chop

def fourier(t_chops, data, n_max = 40):
    cn = []
    cn_negative = []
    f0 = 1/t_chops
    n_freq = int(f_max/f0)
    n_nyquist = len(data) // 2 #max czestotliwosc Nyqist..
    n_freq = min(int(f_max / f0), n_nyquist, n_max)
    
    freqs_array = np.arange(-n_freq, n_freq+1, 1)*f0
    cn.append(furier_projection(t_chops, data, 0))
    
    for n in range(1,n_freq+1): #total freqs will be 2*nfreq +1 (for n=0)
        cn_factor = furier_projection(t_chops, data, n)
        cn.append(cn_factor)
        cn_negative.append(cn_factor.conjugate()) #math stuff - to get rid off imag. part, c-n should be conjucagte of cn
    return (list(reversed(cn_negative)) + cn, freqs_array)

In [104]:
from scipy.io import wavfile
import numpy as np
from math import ceil
#goal is to change time-domain to freq-domain (similiar to .mp3 file)
#firstly, we need to divide all time data to small chops (with lentgh of 1/20hz ~= 0.05s)
file = "wav/orchestra.wav"
sr, data = wavfile.read(file)
f_nyq = sr/2

print("Sample rate:", sr)
print("Shape:", data.shape)
print("dtype:", data.dtype)
print("Time:",round(data.shape[0]/sr, 3), "s")

#normalized and mono file
left = data[:,0]
right = data[:,1]
mono = (left+right)/2
mono = mono/np.max(np.abs(mono))
###
t_chop = 0.05
points_per_chop = int(t_chop*sr)
length = len(left)/sr
n_chops = ceil(length/t_chop)

data_chopped = np.zeros((n_chops,int(ceil(t_chop*sr))))
for i in range(n_chops-1):
    data_chopped[i,:] = mono[i*points_per_chop:i*points_per_chop+points_per_chop]
left_points = int(len(mono)-((n_chops-1)*t_chop)*sr)
data_chopped[-1,:left_points] = mono[-left_points:]
last_index = (n_chops, left_points-1)



Sample rate: 44100
Shape: (659456, 2)
dtype: int16
Time: 14.954 s


In [159]:
import time
import matplotlib.pyplot as plt
Cn =[]
Freqs = []
for i in range(data_chopped.shape[0]):
    cn, freqs = fourier(0.05, data_chopped[i, :], 1000)
    Cn.append(cn) #next chops
    Freqs.append(freqs)
    
print(np.shape(Cn), np.shape(Freqs))

(300, 2001) (300, 2001)


In [ ]:
#skladamy: f(t) = suma cn exp(1j*w*t*n)
f = []
Cn = np.array(Cn, dtype=object)
Freqs = np.array(Freqs, dtype=object)
for chop in range(np.shape(Cn)[0]):
    t = np.linspace(t_chop*chop,t_chop*(chop+1),int(sr*t_chop))
    freq = Freqs[chop]
    cn = Cn[chop]
    omegas = np.array(2*np.pi*freq, dtype=np.float64)
    phase = 1j*np.outer(omegas, t)
    t = np.array(t, dtype=np.float64)


    f.append(np.sum(np.exp(phase) * cn[:, None], axis=0))

In [ ]:
audio = np.real(np.concatenate(f))
audio = audio / np.max(np.abs(audio))   # NORMALIZACJA
audio = (audio * 32767).astype(np.int16)

wavfile.write("wav/reconstructed.wav", sr, audio)

C:\Users\qmiko\AppData\Local\Temp\ipykernel_6092\3787689741.py:3: ComplexWarning: Casting complex values to real discards the imaginary part
  audio = (audio * 32767).astype(np.int16)
